# 03: Validate the coder's sheet
Run after every coding batch. Checks codebook values and the conditional-field rules. Downstream notebooks refuse to run while errors remain.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 80)

In [2]:
from io_utils import load_coder_sheet, FROZEN, TABLES
from coding_schema import is_coded, validation_errors, blocking
sheet = load_coder_sheet()
frame = pd.read_csv(FROZEN / "sample_frame.tsv", sep="\t", dtype=str)
assert sheet.review_id.is_unique and set(sheet.pmid) == set(frame.pmid), "sheet rows altered"
merged = sheet.merge(frame[["review_id", "pmid"]], on="review_id", suffixes=("", "_frame"))
assert (merged.pmid == merged.pmid_frame).all(), "review_id/pmid mismatch"
sheet["coded"] = [is_coded(r) for _, r in sheet.iterrows()]
coded = sheet[sheet.coded].copy()
coded["errors"] = [validation_errors(r) for _, r in coded.iterrows()]
problems = coded[coded.errors.map(bool).astype(bool)]
n_block = int(coded.errors.map(lambda e: bool(blocking(e))).sum())
problems[["review_id", "pmid", "arm", "errors"]].to_csv(TABLES / "coding_validation_errors.tsv", sep="\t", index=False)
print(sheet.groupby("arm").coded.agg(["sum", "size"]).rename(columns={"sum": "coded", "size": "total"}))
print("priority 10% coded:", int(sheet.loc[sheet.priority_10pct == "1", "coded"].sum()), "/ 75")
print("records with blocking errors:", n_block, "| with warnings only:", len(problems) - n_block)
problems[["review_id", "pmid", "errors"]].head(30)

     coded  total
arm              
A      350    350
B      397    398
priority 10% coded: 75 / 75
records with blocking errors: 0 | with warnings only: 15


,review_id,pmid,errors
111,A303,40970514,"[warning: scale required for NC/D, warning: observation required for NC/D, w..."
163,A345,37995077,"[warning: scale required for NC/D, warning: observation required for NC/D, w..."
176,A161,19318942,"[warning: scale required for NC/D, warning: observation required for NC/D, w..."
354,B240,20400446,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
363,B060,7723738,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
381,B151,16204590,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
384,B317,38804500,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
390,B223,21953681,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
392,B241,25063353,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
395,B295,20440547,"[warning: level must be empty outside NC/D, warning: scale must be empty out..."
